In [ ]:
"""
Data Quality - Completeness Check Task
Checks for NULL/empty values in configured columns
"""
from pyspark.sql import functions as F
import json

# Get parameters from notebook widgets
dbutils.widgets.text("project_id", "")
dbutils.widgets.text("project_name", "")
dbutils.widgets.text("job_name", "")
dbutils.widgets.text("catalog", "dbr_uat01_gdm")
dbutils.widgets.text("schema", "db_gdm_data_quality_app")
dbutils.widgets.text("results_schema", "db_gdm_dq_results")
dbutils.widgets.text("columns", "[]")
dbutils.widgets.text("extra_empty_values", "[]")

PROJECT_ID = dbutils.widgets.get("project_id")
PROJECT_NAME = dbutils.widgets.get("project_name")
JOB_NAME = dbutils.widgets.get("job_name")
CATALOG = dbutils.widgets.get("catalog")
SCHEMA = dbutils.widgets.get("schema")
RESULTS_SCHEMA = dbutils.widgets.get("results_schema")
COLUMNS_TO_CHECK = json.loads(dbutils.widgets.get("columns"))
EXTRA_EMPTY_VALUES = json.loads(dbutils.widgets.get("extra_empty_values"))

# Sanitize names for table naming
import re
def sanitize_name(name):
    return re.sub(r'[^a-zA-Z0-9_]', '_', name).lower()
COLUMNS_TO_CHECK = [sanitize_name(c) for c in COLUMNS_TO_CHECK]
project_name_clean = sanitize_name(PROJECT_NAME) if PROJECT_NAME else f"project_{PROJECT_ID}"
job_name_clean = sanitize_name(JOB_NAME) if JOB_NAME else f"job_{PROJECT_ID}"

print(f"=== Data Quality - Completeness Check ===")
print(f"Project: {PROJECT_NAME} (ID: {PROJECT_ID})")
print(f"Job: {JOB_NAME}")
print(f"Columns to check: {COLUMNS_TO_CHECK}")
print(f"Extra empty values: {EXTRA_EMPTY_VALUES}")

# Read input data - now from results schema
input_table = f"{CATALOG}.{RESULTS_SCHEMA}.dq_{project_name_clean}_{job_name_clean}_input"
print(f"Reading from: {input_table}")
df_input = spark.table(input_table)

total_rows = df_input.count()
print(f"Total rows: {total_rows}")

# Build case-insensitive column lookup (handles mixed-case input tables)
actual_columns = {c.lower(): c for c in df_input.columns}

# Start with row_id only
df_checks = df_input.select("row_id")

# Create completeness check for each column
for col_name in COLUMNS_TO_CHECK:
    # Resolve to actual column name in the DataFrame (case-insensitive)
    actual_col = actual_columns.get(col_name.lower(), col_name)
    if actual_col != col_name:
        print(f"  Resolved '{col_name}' → '{actual_col}' (case match)")
    check_col_name = f"{col_name}_completeness_check"

    print(f"Creating check for column: {actual_col} → {check_col_name}")

    # Get column data type
    col_dtype = dict(df_input.dtypes)[actual_col]
    print(f"  Column type: {col_dtype}")

    # Base condition: not null
    condition = F.col(actual_col).isNotNull()

    # Add empty/whitespace-only string check for string/varchar types.
    # F.trim() ensures values like "   " are treated as empty, matching
    # what the UI describes as "whitespace-only = incomplete".
    if col_dtype.lower() in ['string', 'varchar']:
        condition = condition & (F.trim(F.col(actual_col)) != "")

        # Add extra empty values check if configured (only for string columns).
        # Trim both sides so comparisons are whitespace-insensitive.
        if EXTRA_EMPTY_VALUES:
            for empty_val in EXTRA_EMPTY_VALUES:
                condition = condition & (F.trim(F.col(actual_col)) != empty_val.strip())

    # Add check column: 1 if valid, 0 if invalid
    df_checks = df_checks.join(
        df_input.select("row_id", actual_col).withColumn(
            check_col_name,
            F.when(condition, 1).otherwise(0)
        ).select("row_id", check_col_name),
        "row_id",
        "left"
    )

# Add execution timestamp
df_checks = df_checks.withColumn("execution_timestamp", F.current_timestamp())

# Output table name - using results schema with meaningful naming
output_table = f"{CATALOG}.{RESULTS_SCHEMA}.dq_{project_name_clean}_{job_name_clean}_completeness"

print(f"Writing to: {output_table}")

# Write to Unity Catalog
df_checks.write \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(output_table)

print(f"✅ Successfully wrote {df_checks.count()} rows to {output_table}")

# Display statistics
print("\nCompleteness Statistics:")
for col_name in COLUMNS_TO_CHECK:
    check_col_name = f"{col_name}_completeness_check"
    valid_count = df_checks.filter(F.col(check_col_name) == 1).count()
    invalid_count = df_checks.filter(F.col(check_col_name) == 0).count()
    completeness_pct = (valid_count / total_rows * 100) if total_rows > 0 else 0
    print(f"  {col_name}: {completeness_pct:.2f}% ({valid_count}/{total_rows} valid)")